In [1]:
"""
Modelo Prophet para previsão de visualizações diárias do Portal TRT18
Mesma metodologia aplicada ao ARIMA, SARIMAX e SVR.
"""
 
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
 
warnings.filterwarnings('ignore')
 
# ─────────────────────────────────────────────
# 0. Diretórios
# ─────────────────────────────────────────────
os.makedirs('../resultados', exist_ok=True)
 
# ─────────────────────────────────────────────
# 1. Leitura e preparação da base comum
# ─────────────────────────────────────────────
df = pd.read_csv('../dados/trafego_tratado.csv')
df['Data'] = pd.to_datetime(df['Data'])
df = df.sort_values('Data').reset_index(drop=True)
 
# Variável-alvo
TARGET = 'Visualizações'
 
# ─────────────────────────────────────────────
# 2. Divisão treino/teste (80/20 sequencial)
# ─────────────────────────────────────────────
n = len(df)
split = int(n * 0.80)
df_train = df.iloc[:split].copy()
df_test  = df.iloc[split:].copy()
 
print(f"Total de registros : {n}")
print(f"Treino             : {len(df_train)}  ({df_train['Data'].min().date()} → {df_train['Data'].max().date()})")
print(f"Teste              : {len(df_test)}   ({df_test['Data'].min().date()} → {df_test['Data'].max().date()})")
 
# ─────────────────────────────────────────────
# 3. Preparação da base no formato Prophet
#    O Prophet exige colunas 'ds' e 'y'.
# ─────────────────────────────────────────────
 
# Colunas de calendário disponíveis na base tratada
# (sessões e usuários ativos são excluídas — informação contemporânea)
REGRESSORES = [
    'recesso_judiciario',
    'feriado_nacional_fixo',
    'carnaval',
    'quarta_cinzas',
    'sexta_paixao',
    'corpus_christi',
    'data_especifica_judiciario',
    'ponto_facultativo_emenda',
]
 
def prep_prophet(df_input):
    """Renomeia e adiciona regressores exógenos."""
    out = df_input[['Data', TARGET] + REGRESSORES].copy()
    out = out.rename(columns={'Data': 'ds', TARGET: 'y'})
    return out
 
df_train_p = prep_prophet(df_train)
df_test_p  = prep_prophet(df_test)
 
# ─────────────────────────────────────────────
# 4. Treinamento do modelo
# ─────────────────────────────────────────────
# Sazonalidade semanal já embutida no Prophet por padrão.
# Sazonalidade anual também ativada — o período amostral (~3 anos)
# é suficiente para estimá-la.
# Sazonalidade diária desativada: dados são diários agregados, não
# intraday, portanto não existe padrão dentro do dia.
 
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',  # volume varia proporcionalmente ao nível
)
 
# Adiciona regressores exógenos — idênticos aos usados no SARIMAX
for col in REGRESSORES:
    model.add_regressor(col)
 
model.fit(df_train_p)
 
# ─────────────────────────────────────────────
# 5. Previsão sobre o conjunto de teste
# ─────────────────────────────────────────────
future = df_test_p[['ds'] + REGRESSORES].copy()
forecast = model.predict(future)
 
y_real = df_test_p['y'].values
y_pred = forecast['yhat'].values
 
# ─────────────────────────────────────────────
# 6. Métricas de avaliação
# ─────────────────────────────────────────────
mae  = mean_absolute_error(y_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_real, y_pred))
r2   = r2_score(y_real, y_pred)
 
print("\n── Métricas (conjunto de teste) ──────────────")
print(f"MAE   : {mae:,.0f}")
print(f"RMSE  : {rmse:,.0f}")
print(f"R²    : {r2:.4f}")
 
# ─────────────────────────────────────────────
# 7. Gráfico: valores reais vs. previstos
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
 
ax.plot(df_test_p['ds'], y_real, label='Real', color='steelblue', linewidth=1.2)
ax.plot(df_test_p['ds'], y_pred, label='Previsto (Prophet)',
        color='darkorange', linewidth=1.2, linestyle='--')
 
ax.set_title('Prophet — Visualizações diárias: valores reais vs. previstos (conjunto de teste)',
             fontsize=12)
ax.set_xlabel('Data')
ax.set_ylabel('Visualizações')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)
 
plt.tight_layout()
plt.savefig('../resultados/05_prophet_real_vs_previsto.png', dpi=150, bbox_inches='tight')
plt.close()
print("Gráfico salvo em ../resultados/05_prophet_real_vs_previsto.png")
 
# ─────────────────────────────────────────────
# 8. Gráfico de comparação com todos os modelos
#    (adicionado após rodar ARIMA, SARIMAX, SVR)
# ─────────────────────────────────────────────
 
# Resultados dos modelos anteriores (coletados dos notebooks correspondentes)
resultados_todos = {
    'ARIMA'  : {'MAE':  6711, 'RMSE': 11853, 'R2': 0.50},
    'SARIMAX': {'MAE':  6488, 'RMSE': 10279, 'R2': 0.63},
    'Prophet': {'MAE':  round(mae),  'RMSE': round(rmse),  'R2': round(r2, 4)},
    'SVR'    : {'MAE':  3179, 'RMSE':  4636, 'R2': 0.92},
}
 
models  = list(resultados_todos.keys())
maes    = [v['MAE']  for v in resultados_todos.values()]
rmses   = [v['RMSE'] for v in resultados_todos.values()]
r2s     = [v['R2']   for v in resultados_todos.values()]
 
colors  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
 
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
 
# MAE
axes[0].bar(models, maes, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('MAE (menor é melhor)', fontsize=11)
axes[0].set_ylabel('Visualizações')
for i, v in enumerate(maes):
    axes[0].text(i, v + 80, f'{v:,.0f}', ha='center', va='bottom', fontsize=9)
axes[0].set_ylim(0, max(maes) * 1.15)
 
# RMSE
axes[1].bar(models, rmses, color=colors, edgecolor='white', linewidth=0.8)
axes[1].set_title('RMSE (menor é melhor)', fontsize=11)
axes[1].set_ylabel('Visualizações')
for i, v in enumerate(rmses):
    axes[1].text(i, v + 150, f'{v:,.0f}', ha='center', va='bottom', fontsize=9)
axes[1].set_ylim(0, max(rmses) * 1.15)
 
# R²
axes[2].bar(models, r2s, color=colors, edgecolor='white', linewidth=0.8)
axes[2].set_title('R² (maior é melhor)', fontsize=11)
axes[2].set_ylabel('R²')
for i, v in enumerate(r2s):
    axes[2].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
axes[2].set_ylim(0, 1.1)
 
for ax in axes:
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)
 
fig.suptitle('Comparação de desempenho — modelos avaliados (conjunto de teste)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../resultados/06_comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.close()
print("Gráfico salvo em ../resultados/06_comparacao_modelos.png")
 
print("\nScript concluído.")
 

Importing plotly failed. Interactive plots will not work.


Total de registros : 1020
Treino             : 816  (2023-07-01 → 2025-09-23)
Teste              : 204   (2025-09-24 → 2026-04-15)


00:57:39 - cmdstanpy - INFO - Chain [1] start processing
00:57:40 - cmdstanpy - INFO - Chain [1] done processing



── Métricas (conjunto de teste) ──────────────
MAE   : 7,826
RMSE  : 9,742
R²    : 0.6637
Gráfico salvo em ../resultados/05_prophet_real_vs_previsto.png
Gráfico salvo em ../resultados/06_comparacao_modelos.png

Script concluído.
